# Lesson 74 · Make every wrong word give up its clues — ASR error analysis, from a WER number to what to fix

> **Where this fits**: this lesson uses code-level counting and a table of error patterns to break WER down into a list of improvements you can act on.

**Learning goals**
- Pull the actual words behind each S/D/I error out of the alignment path
- Count the most frequent error pairs (substitution pairs)
- Spot which kinds of words are hardest for ASR (numbers, proper names)


← **Previous lesson** · [L73 · Evaluating with WER](L73_wer_eval.ipynb)

> Last lesson covered **evaluating with WER**: the word error rate (insertions / deletions / substitutions) and sentence-by-sentence analysis compared with jiwer.
> This lesson explores **ASR error analysis**.

In [ ]:
import numpy as np
from collections import Counter

In [ ]:
# Reuse the alignment function from L67
def alignment(ref_str, hyp_str):
    """Return a list of word-level alignment operations: (op, ref_word, hyp_word). op ∈ {M,S,I,D}"""
    # Lowercase everything before comparing, so case differences don't skew the error analysis
    a, b = ref_str.lower().split(), hyp_str.lower().split()
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1,m+1):
        for j in range(1,n+1):
            if a[i-1]==b[j-1]: dp[i][j]=dp[i-1][j-1]
            else: dp[i][j]=1+min(dp[i-1][j],dp[i][j-1],dp[i-1][j-1])
    ops,i,j=[],m,n
    while i>0 or j>0:
        if i>0 and j>0 and a[i-1]==b[j-1]:
            ops.append(('M',a[i-1],b[j-1])); i-=1; j-=1
        elif i>0 and j>0 and dp[i][j]==1+dp[i-1][j-1]:
            ops.append(('S',a[i-1],b[j-1])); i-=1; j-=1
        elif j>0 and dp[i][j]==1+dp[i][j-1]:
            ops.append(('I','-',b[j-1])); j-=1
        else:
            ops.append(('D',a[i-1],'-')); i-=1
    return list(reversed(ops))


In [ ]:
def wer(ref_str, hyp_str):
    """Word error rate = (S+D+I) / |ref|, counting each operation with alignment()"""
    # ✏️ TODO: ops = alignment(ref_str, hyp_str)
    # ✏️ TODO: count how many S, D and I there are
    # ✏️ TODO: return (S+D+I) / len(ref_str.split())
    raise NotImplementedError('please implement wer')


In [ ]:
# Check the WER implementation
try:
    result = wer("the cat sat", "the cat sat")
    assert result is not None, "⬜ wer not implemented yet"
    assert abs(result - 0.0) < 1e-9, f"identical sentences should have WER 0, got {result}"

    result2 = wer("the cat sat", "the dog sat")
    assert abs(result2 - 1/3) < 0.01, f"one different word should give WER ≈ 0.33, got {result2}"

    print(f"✅ WER check passed (identical={result:.3f}, one word different={result2:.3f})")
except (NotImplementedError, TypeError):
    print("⬜ wer not implemented yet")

## 10 (ref, hyp) sample pairs

In [ ]:
PAIRS = [
    ("the quick brown fox jumps over the lazy dog",
     "the quick brown fox jump over the lazy dog"),
    ("she sells seashells by the seashore",
     "she cells seashells by the seashore"),
    ("call nine one one immediately",
     "call 911 immediately"),
    ("the meeting is scheduled for three pm on friday",
     "the meeting is scheduled for 3 pm on friday"),
    ("artificial intelligence is transforming healthcare",
     "artificial intelligence is transforming health care"),
    ("please send the report to john smith",
     "please send the report to john's myth"),
    ("the temperature is twenty three degrees celsius",
     "the temperature is 23 degrees celsius"),
    ("i would like a cup of coffee please",
     "i would like a cup of coffee"),
    ("deep learning models require large amounts of training data",
     "deep learning models require large amounts of training data"),
    ("natural language processing has many applications",
     "natural language process has many application"),
]

print(f'{"#":>2}  {"WER":>5}  REF')
print('-' * 55)
try:
    for i, (ref, hyp) in enumerate(PAIRS):
        w = wer(ref, hyp)
        print(f'{i:>2}  {w:>5.1%}  {ref[:45]}')
except (NotImplementedError, TypeError):
    print('⬜ wer not implemented yet — finish the TODO above, then re-run this cell')

## Counting the errors

In [ ]:
sub_counter = Counter()
del_counter = Counter()
ins_counter = Counter()

for ref, hyp in PAIRS:
    for op, rw, hw in alignment(ref, hyp):
        if op == 'S':
            sub_counter[(rw, hw)] += 1
        elif op == 'D':
            del_counter[rw] += 1
        elif op == 'I':
            ins_counter[hw] += 1

print('=== Most common substitutions (ref→hyp) ===')
for (r, h), cnt in sub_counter.most_common(8):
    print(f'  {r:15s} → {h:15s}  ×{cnt}')

print()
print('=== Most common deletions ===')
for word, cnt in del_counter.most_common(5):
    print(f'  deleted: {word:15s}  ×{cnt}')

print()
print('=== Most common insertions ===')
for word, cnt in ins_counter.most_common(5):
    print(f'  inserted: {word:15s}  ×{cnt}')

## Summary of error patterns

| Error type | Example | Root cause | Way to improve |
|---|---|---|---|
| Number/word mix-up | nine one one → 911 | Inconsistent formatting in the training data | Post-processing normalization |
| Similar-sounding word swapped | sells → cells | Similar acoustic features | More context / LM |
| Proper names | john smith → john's myth | OOV (out-of-vocabulary) words | Hotword boosting / fine-tuning |
| Final word dropped | coffee please → coffee | Low energy at the end of the sentence | Better VAD |

**Next step, L75**: draw the pipeline these errors come out of — CTC alignment paths and cross-attention between audio frames and tokens.

## Quantitative view: using S/D/I to aim your improvements

**WER decomposition**: `WER = (S + D + I) / N_ref`

| Dominant error type | What it means | Improve first |
|---|---|---|
| S (substitution) dominates | Similar-sounding words get confused | Bigger acoustic model / domain LM |
| D (deletion) dominates | Sentence ends / weakly spoken words get missed | Tune the VAD / augment the training data |
| I (insertion) dominates | Noise gets recognized as words | Denoising front end / adjust thresholds |

What error analysis buys you: it turns WER from a black-box number into a list of improvements you can act on.

In [ ]:
# Check: the S/D/I counts add up to the total number of edit operations, so the WER decomposition is self-consistent
total_S = sum(sub_counter.values())
total_D = sum(del_counter.values())
total_I = sum(ins_counter.values())
total_ref_words = sum(len(ref.split()) for ref, _ in PAIRS)

wer_val = (total_S + total_D + total_I) / total_ref_words
print(f'Substitutions (S): {total_S}')
print(f'Deletions (D): {total_D}')
print(f'Insertions (I): {total_I}')
print(f'Total reference words (N_ref): {total_ref_words}')
print(f'WER = ({total_S}+{total_D}+{total_I}) / {total_ref_words} = {wer_val:.3f}')

assert total_S + total_D + total_I > 0, 'no errors detected at all'
assert 0 < wer_val < 1, f'WER={wer_val:.3f} is outside a sensible range'
print('✅ S/D/I counts are self-consistent; the WER decomposition is correct')

In [ ]:
# Q4: numerical check of S/D/I
try:
    wer_val = wer('the cat sat on the mat', 'the cat sits on mat')
    ops = alignment('the cat sat on the mat', 'the cat sits on mat')
    S = sum(1 for op,*_ in ops if op=='S')
    D = sum(1 for op,*_ in ops if op=='D')
    I = sum(1 for op,*_ in ops if op=='I')
    ref_len = len('the cat sat on the mat'.split())
    expected = (S + D + I) / ref_len
    assert abs(wer_val - expected) < 1e-9, f'WER does not match the S/D/I counts: {wer_val:.3f} vs {expected:.3f}'
    print(f'✅ S/D/I check passed  S={S} D={D} I={I}  WER={wer_val:.3f}')
except (NotImplementedError, TypeError):
    print('⬜ wer not implemented yet')


## Wrap-up

You can now break WER down into its three kinds of error — S (substitution), D (deletion) and I (insertion) — and, by counting words, trace each kind back to the specific words involved and to a direction for improvement (acoustic model / VAD / denoising / language model). The error counting in this lesson's `wer()` and `alignment()` maps directly onto the core logic of `wer` / `corpus_wer` in `aurora.speech.metrics` (actual path: `src/aurora/speech/metrics.py`).

Note: the WER here is a **corpus-level WER** (add up S/D/I and N_ref over all sentences first, then divide once), not the average of per-sentence WERs. The corpus-level convention is closer to how ASR benchmarks are usually scored, because longer sentences are automatically weighted by their number of reference words.

Next lesson (L75) draws the alignment at the heart of this chain — CTC paths and the cross-attention between audio frames and output tokens — so you can see where in time each output token comes from.

In [ ]:
# ── Standalone maths assertions: core properties of alignment (uses the reference implementation, no student code needed) ──

# 1. Exact match → all M operations, edit_distance=0
ops_id = alignment("a b c", "a b c")
assert all(op == 'M' for op, *_ in ops_id), "an exact match should be all M operations"
edit_id = sum(1 for op, *_ in ops_id if op != 'M')
assert edit_id == 0, f"edit distance for an exact match should be 0, got {edit_id}"
print(f"1 ✅  identity: alignment('a b c','a b c') → all M, edit distance = 0")

# 2. One word substituted → S=1
ops_s = alignment("a b c", "a x c")
S2 = sum(1 for op, *_ in ops_s if op == 'S')
assert S2 == 1, f"'b→x' should be 1 substitution, got S={S2}"
print(f"2 ✅  substitution: 'b→x', S={S2}")

# 3. One word deleted → D=1
ops_d = alignment("a b c", "a c")
D3 = sum(1 for op, *_ in ops_d if op == 'D')
assert D3 == 1, f"deleting 'b' should give D=1, got D={D3}"
print(f"3 ✅  deletion: deleted 'b', D={D3}")

# 4. One word inserted → I=1
ops_i = alignment("a c", "a b c")
I4 = sum(1 for op, *_ in ops_i if op == 'I')
assert I4 == 1, f"inserting 'b' should give I=1, got I={I4}"
print(f"4 ✅  insertion: inserted 'b', I={I4}")

# 5. WER formula check: ref="a b c d", hyp="x b c" → S=1 ('a→x'), D=1 ('d'), I=0, N=4 → WER=0.5
ops_wer = alignment("a b c d", "x b c")
S5 = sum(1 for op, *_ in ops_wer if op == 'S')
D5 = sum(1 for op, *_ in ops_wer if op == 'D')
I5 = sum(1 for op, *_ in ops_wer if op == 'I')
wer5 = (S5 + D5 + I5) / 4
assert abs(wer5 - 0.5) < 1e-9, f"WER should be 0.5, got {wer5:.4f}"
print(f"5 ✅  WER formula: S={S5},D={D5},I={I5},N=4 → WER={wer5:.2f}")


In [ ]:
# ✏️ Self-review for this lesson
l74_review = {
    "wer_impl":              None,  # wer() correctly counts S/D/I with alignment(), and both checks pass? True/False
    "sdi_three_types":       None,  # can tell S/D/I apart and name a typical cause of each (similar-sounding words / low energy at sentence end / noise)? True/False
    "counter_analysis":      None,  # can use Counter to trace errors back to specific words (most common substitutions/deletions/insertions)? True/False
    "error_fix_mapping":     None,  # remember what to improve first when S, D or I dominates (LM / VAD / denoising)? True/False
    "sdi_self_consistent":   None,  # understand that S+D+I = total edit operations, and the WER-decomposition self-consistency check passes? True/False
}

unfilled = [k for k, v in l74_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l74_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L74 all clear! On to L75: the ASR pipeline drawn')

---

→ **Next lesson** · [L75 · The ASR pipeline drawn](L75_visual_asr.ipynb)

> Next lesson covers **the ASR pipeline drawn**: visualizing the path waveform → spectrogram → tokens → text.